# LeetCode 1629 — Slowest Key bằng Python

**Link đề:** https://leetcode.com/problems/slowest-key/
**Độ khó:** Easy · **Chủ đề:** Array, String

```python
class Solution:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        ...
```

## Mục tiêu học
1. Đọc đúng dữ liệu thời gian: từ **thời điểm nhả phím** suy ra **thời lượng giữ phím** `releaseTimes[i] - releaseTimes[i-1]` (phím đầu: `releaseTimes[0]`).
2. Nắm mẫu "**tìm max có luật phá hòa (tie-break)**": bằng thời lượng thì chọn phím **lớn hơn theo thứ tự từ điển**.
3. Học so sánh chuỗi `'c' > 'b'`, so sánh **tuple** `(duration, key)`, `zip`, `max(..., key=...)`.
4. Viết 4 cách: tạo list thời lượng rồi tìm max → một lượt duyệt `O(1)` bộ nhớ → `max` trên tuple → dict max theo từng phím.
5. Trace từng lần bấm, test đối chiếu và luyện biến thể (phím nhanh nhất, tổng thời gian mỗi phím...).

## 1. Đề bài

Một người thử bàn phím bấm lần lượt `n` phím. Cho:
- `keysPressed` (chuỗi dài `n`): `keysPressed[i]` là phím bấm ở lần thứ `i`.
- `releaseTimes` (list **tăng dần** dài `n`): `releaseTimes[i]` là thời điểm **nhả** phím thứ `i`.

Phím `0` được bấm ở thời điểm `0`; mỗi phím sau được bấm **đúng lúc** phím trước vừa nhả. Vậy:
- Thời lượng lần `0`: `releaseTimes[0]`.
- Thời lượng lần `i ≥ 1`: `releaseTimes[i] - releaseTimes[i-1]`.

Trả về **phím** của lần bấm có thời lượng **dài nhất**. Nếu nhiều lần bấm cùng dài nhất, trả về phím **lớn nhất theo thứ tự từ điển** (`'z'` > ... > `'a'`).

**Ví dụ 1:** `releaseTimes = [9,29,49,50]`, `keysPressed = "cbcd"` → `"c"`

| i | phím | thời lượng |
|:-:|:-:|:-:|
| 0 | c | 9 |
| 1 | b | 29 − 9 = 20 |
| 2 | c | 49 − 29 = 20 |
| 3 | d | 50 − 49 = 1 |

Dài nhất là 20 (của `b` và `c`) → hòa → `'c' > 'b'` → trả `"c"`.

**Ví dụ 2:** `releaseTimes = [12,23,36,46,62]`, `keysPressed = "spuda"` → `"a"` (thời lượng `12, 11, 13, 10, 16`, lớn nhất 16 là của `a`).

**Constraints:** `2 <= n <= 1000`; `1 <= releaseTimes[i] <= 10^9`; `releaseTimes[i] < releaseTimes[i+1]`; chữ thường.

Lưu ý: cùng một phím có thể được bấm nhiều lần với thời lượng khác nhau — ta so **từng lần bấm**, không cộng dồn theo phím.

In [1]:
examples = [([9, 29, 49, 50], "cbcd", "c"), ([12, 23, 36, 46, 62], "spuda", "a")]
for times, keys, expected in examples:
    print(times, repr(keys), "-> kỳ vọng", repr(expected))

[9, 29, 49, 50] 'cbcd' -> kỳ vọng 'c'
[12, 23, 36, 46, 62] 'spuda' -> kỳ vọng 'a'


## 2. Nền tảng Python

### 2.1. Từ thời điểm sang thời lượng — hiệu hai phần tử kề nhau

Thời lượng = "mốc hiện tại − mốc trước". Mốc trước của lần `0` là `0`. Một mẹo: thêm `0` vào đầu list rồi lấy hiệu kề nhau.

In [2]:
times = [9, 29, 49, 50]
durations = [times[0]] + [times[i] - times[i - 1] for i in range(1, len(times))]
print("cách 1:", durations)
padded = [0] + times
print("cách 2 (đệm 0):", [padded[i + 1] - padded[i] for i in range(len(times))])

cách 1: [9, 20, 20, 1]
cách 2 (đệm 0): [9, 20, 20, 1]


### 2.2. `zip` — ghép từng cặp phần tử

`zip(a, b)` trả về các cặp `(a[0], b[0]), (a[1], b[1]), ...`. `zip(padded, times)` ghép "mốc trước" với "mốc hiện tại" → tiện tính hiệu mà không cần chỉ số.

In [3]:
times = [9, 29, 49, 50]
for prev, cur in zip([0] + times, times):
    print(f"prev={prev:>2}, cur={cur:>2} -> thời lượng {cur - prev}")
print(list(zip("cbcd", [9, 20, 20, 1])))

prev= 0, cur= 9 -> thời lượng 9
prev= 9, cur=29 -> thời lượng 20
prev=29, cur=49 -> thời lượng 20
prev=49, cur=50 -> thời lượng 1
[('c', 9), ('b', 20), ('c', 20), ('d', 1)]


### 2.3. So sánh chuỗi theo thứ tự từ điển

Với chữ thường, `'a' < 'b' < ... < 'z'` (so theo mã `ord`). `max('b', 'c')` là `'c'`.

In [4]:
print('c' > 'b', 'a' > 'z', max('b', 'c'), ord('b'), ord('c'))

True False c 98 99


### 2.4. So sánh tuple: so phần tử đầu, hòa thì so phần tử sau

`(20, 'c') > (20, 'b')` vì phần đầu bằng nhau, phần sau `'c' > 'b'`. `(20, 'a') > (9, 'z')` vì `20 > 9` (không cần xét phần sau).

Đây đúng là luật của đề: **ưu tiên thời lượng, hòa thì so phím** → gói thành tuple `(duration, key)` rồi lấy `max`!

In [5]:
print((20, 'c') > (20, 'b'))
print((20, 'a') > (9, 'z'))
print(max([(9, 'c'), (20, 'b'), (20, 'c'), (1, 'd')]))

True
True
(20, 'c')


## 3. Cách 1 — Trực giác: tạo list thời lượng, tìm max, rồi lọc phím

Ba bước tách bạch, dễ kiểm tra:
1. Tính list `durations`.
2. `longest = max(durations)`.
3. Trong các lần bấm có `duration == longest`, lấy phím lớn nhất.

In [6]:
def slowest_key_lists(releaseTimes: list[int], keysPressed: str) -> str:
    n = len(releaseTimes)
    durations = [releaseTimes[0]] + [releaseTimes[i] - releaseTimes[i - 1] for i in range(1, n)]
    longest = max(durations)
    candidates = [keysPressed[i] for i in range(n) if durations[i] == longest]
    return max(candidates)

for times, keys, expected in examples:
    print(slowest_key_lists(times, keys), "| kỳ vọng", expected)

c | kỳ vọng c
a | kỳ vọng a


## 4. Nút thắt và insight — làm trong **một lượt**, không cần list

Cách 1 duyệt 3 lần và dùng thêm `O(n)` bộ nhớ. Thực ra chỉ cần giữ **kỷ lục hiện tại**: `best_dur` và `best_key`. Với mỗi lần bấm `(d, k)`:

> Thay kỷ lục ⟺ `d > best_dur` **hoặc** (`d == best_dur` **và** `k > best_key`).

Đó chính là hint thứ 3 của LeetCode. Và điều kiện này tương đương `(d, k) > (best_dur, best_key)` — so sánh tuple.

In [7]:
best_dur, best_key = 9, "c"
for d, k in [(20, "b"), (20, "c"), (1, "d")]:
    replace = d > best_dur or (d == best_dur and k > best_key)
    print(f"xét ({d}, {k!r}) vs kỷ lục ({best_dur}, {best_key!r}) -> thay? {replace} | tuple so sánh: {(d, k) > (best_dur, best_key)}")
    if replace:
        best_dur, best_key = d, k
print("kết quả:", best_key)

xét (20, 'b') vs kỷ lục (9, 'c') -> thay? True | tuple so sánh: True
xét (20, 'c') vs kỷ lục (20, 'b') -> thay? True | tuple so sánh: True
xét (1, 'd') vs kỷ lục (20, 'c') -> thay? False | tuple so sánh: False
kết quả: c


## 5. Lời giải chính — một lượt duyệt (giảng từng đoạn)

### 5.1. Đoạn 1: khởi tạo kỷ lục bằng lần bấm đầu tiên
`best_dur = releaseTimes[0]`, `best_key = keysPressed[0]`. (Lần đầu có mốc trước là 0.)

### 5.2. Đoạn 2: duyệt `i` từ `1` tới `n-1`, tính `d = releaseTimes[i] - releaseTimes[i-1]`

### 5.3. Đoạn 3: áp luật thay kỷ lục

In [8]:
def slowest_key_one_pass(releaseTimes: list[int], keysPressed: str) -> str:
    best_dur = releaseTimes[0]
    best_key = keysPressed[0]
    for i in range(1, len(releaseTimes)):
        d = releaseTimes[i] - releaseTimes[i - 1]
        k = keysPressed[i]
        if d > best_dur or (d == best_dur and k > best_key):
            best_dur, best_key = d, k
    return best_key

for times, keys, expected in examples:
    print(slowest_key_one_pass(times, keys), "| kỳ vọng", expected)

c | kỳ vọng c
a | kỳ vọng a


`best_dur, best_key = d, k` là **gán đồng thời** hai biến (tuple unpacking) — gọn và tránh quên cập nhật một biến.

In [9]:
x, y = 1, 2
x, y = y, x        # hoán đổi cũng dùng cùng cơ chế
print(x, y)

2 1


## 6. Cách 3 — `max` trên tuple `(duration, key)`

Ghép thời lượng và phím thành tuple, lấy `max`, trả phần tử thứ hai `[1]`. Luật phá hòa có sẵn nhờ so sánh tuple.

In [10]:
def slowest_key_tuple(releaseTimes: list[int], keysPressed: str) -> str:
    prev = [0] + releaseTimes[:-1]
    return max(zip((cur - p for cur, p in zip(releaseTimes, prev)), keysPressed))[1]

print([slowest_key_tuple(t, k) for t, k, _ in examples])

['c', 'a']


Đọc từ trong ra ngoài:
- `releaseTimes[:-1]` bỏ phần tử cuối; `prev = [0] + ...` là "mốc trước" của từng lần bấm.
- `(cur - p for ...)` là generator thời lượng.
- `zip(thời_lượng, keysPressed)` tạo các tuple `(d, k)`.
- `max(...)` chọn tuple lớn nhất, `[1]` lấy phím.

### 6.1. Biến thể dùng `max(range(n), key=...)`

`key=` cho `max` biết **tiêu chí so sánh**: ở đây so theo tuple `(duration(i), keysPressed[i])`, trả về **chỉ số** tốt nhất.

In [11]:
def slowest_key_index(releaseTimes: list[int], keysPressed: str) -> str:
    def duration(i):
        return releaseTimes[i] - (releaseTimes[i - 1] if i > 0 else 0)
    best_i = max(range(len(releaseTimes)), key=lambda i: (duration(i), keysPressed[i]))
    return keysPressed[best_i]

print([slowest_key_index(t, k) for t, k, _ in examples])
print("max theo độ dài:", max(["ab", "c", "xyz"], key=len))

['c', 'a']
max theo độ dài: xyz


`lambda i: ...` là **hàm vô danh** viết trên một dòng: `lambda i: (duration(i), keysPressed[i])` tương đương `def f(i): return (duration(i), keysPressed[i])`.

## 7. Cách 4 — Dict: thời lượng dài nhất của mỗi phím

Gom theo phím: `longest[k] = max thời lượng của các lần bấm phím k`. Sau đó chọn phím có `(longest[k], k)` lớn nhất. Cách này hơi "thừa" cho bài gốc, nhưng hữu ích khi đề hỏi thêm thống kê theo phím.

In [12]:
def slowest_key_dict(releaseTimes: list[int], keysPressed: str) -> str:
    longest = {}
    prev = 0
    for t, k in zip(releaseTimes, keysPressed):
        d = t - prev
        prev = t
        longest[k] = max(longest.get(k, 0), d)
    return max(longest, key=lambda k: (longest[k], k))

print([slowest_key_dict(t, k) for t, k, _ in examples])

['c', 'a']


`longest.get(k, 0)`: lấy giá trị của `k`, chưa có thì `0` (thời lượng luôn ≥ 1 nên 0 an toàn). `max(dict, key=...)` duyệt qua các **key** của dict.

## 8. TRACE từng bước

Trace ví dụ 1 với cách một lượt:

| i | phím | mốc trước | mốc nhả | d | so với kỷ lục | kỷ lục sau |
|:-:|:-:|:-:|:-:|:-:|---|---|
| 0 | c | 0 | 9 | 9 | khởi tạo | (9, c) |
| 1 | b | 9 | 29 | 20 | 20 > 9 → thay | (20, b) |
| 2 | c | 29 | 49 | 20 | bằng 20, `'c' > 'b'` → thay | (20, c) |
| 3 | d | 49 | 50 | 1 | 1 < 20 → giữ | (20, c) |

Kết quả `"c"`.

In [13]:
def trace_slowest_key(releaseTimes, keysPressed):
    best_dur, best_key = releaseTimes[0], keysPressed[0]
    print(f"i=0 phím={best_key!r} d={best_dur} -> khởi tạo kỷ lục ({best_dur}, {best_key!r})")
    for i in range(1, len(releaseTimes)):
        d = releaseTimes[i] - releaseTimes[i - 1]
        k = keysPressed[i]
        if d > best_dur:
            reason = f"{d} > {best_dur} -> thay"
        elif d == best_dur and k > best_key:
            reason = f"hòa {d}, {k!r} > {best_key!r} -> thay"
        elif d == best_dur:
            reason = f"hòa {d}, {k!r} <= {best_key!r} -> giữ"
        else:
            reason = f"{d} < {best_dur} -> giữ"
        if d > best_dur or (d == best_dur and k > best_key):
            best_dur, best_key = d, k
        print(f"i={i} phím={k!r} d={releaseTimes[i]}-{releaseTimes[i-1]}={d} | {reason} | kỷ lục ({best_dur}, {best_key!r})")
    return best_key

print("Kết quả:", trace_slowest_key([9, 29, 49, 50], "cbcd"))

i=0 phím='c' d=9 -> khởi tạo kỷ lục (9, 'c')
i=1 phím='b' d=29-9=20 | 20 > 9 -> thay | kỷ lục (20, 'b')
i=2 phím='c' d=49-29=20 | hòa 20, 'c' > 'b' -> thay | kỷ lục (20, 'c')
i=3 phím='d' d=50-49=1 | 1 < 20 -> giữ | kỷ lục (20, 'c')
Kết quả: c


Trace có nhánh "hòa nhưng phím nhỏ hơn → giữ" (`"zay"` với thời lượng `5, 5, 5`):

In [14]:
print("Kết quả:", trace_slowest_key([5, 10, 15], "zay"))
print()
print("Kết quả:", trace_slowest_key([12, 23, 36, 46, 62], "spuda"))

i=0 phím='z' d=5 -> khởi tạo kỷ lục (5, 'z')
i=1 phím='a' d=10-5=5 | hòa 5, 'a' <= 'z' -> giữ | kỷ lục (5, 'z')
i=2 phím='y' d=15-10=5 | hòa 5, 'y' <= 'z' -> giữ | kỷ lục (5, 'z')
Kết quả: z

i=0 phím='s' d=12 -> khởi tạo kỷ lục (12, 's')
i=1 phím='p' d=23-12=11 | 11 < 12 -> giữ | kỷ lục (12, 's')
i=2 phím='u' d=36-23=13 | 13 > 12 -> thay | kỷ lục (13, 'u')
i=3 phím='d' d=46-36=10 | 10 < 13 -> giữ | kỷ lục (13, 'u')
i=4 phím='a' d=62-46=16 | 16 > 13 -> thay | kỷ lục (16, 'a')
Kết quả: a


## 9. Code nộp LeetCode

### 9.1. Cách chính — một lượt duyệt

In [15]:
class Solution:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        best_dur, best_key = releaseTimes[0], keysPressed[0]
        for i in range(1, len(releaseTimes)):
            d = releaseTimes[i] - releaseTimes[i - 1]
            if d > best_dur or (d == best_dur and keysPressed[i] > best_key):
                best_dur, best_key = d, keysPressed[i]
        return best_key

print([Solution().slowestKey(t, k) for t, k, _ in examples])   # ['c', 'a']

['c', 'a']


### 9.2. Các cách khác

In [16]:
class SolutionTuple:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        prev = [0] + releaseTimes[:-1]
        return max(zip((c - p for c, p in zip(releaseTimes, prev)), keysPressed))[1]

class SolutionLists:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        durations = [releaseTimes[0]] + [releaseTimes[i] - releaseTimes[i - 1] for i in range(1, len(releaseTimes))]
        longest = max(durations)
        return max(k for d, k in zip(durations, keysPressed) if d == longest)

class SolutionDict:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        longest, prev = {}, 0
        for t, k in zip(releaseTimes, keysPressed):
            longest[k] = max(longest.get(k, 0), t - prev)
            prev = t
        return max(longest, key=lambda k: (longest[k], k))

for cls in [SolutionTuple, SolutionLists, SolutionDict]:
    print(cls.__name__, [cls().slowestKey(t, k) for t, k, _ in examples])

SolutionTuple ['c', 'a']
SolutionLists ['c', 'a']
SolutionDict ['c', 'a']


## 10. Vì sao đúng? (invariant)

Định nghĩa thứ tự giữa các lần bấm: `(d₁, k₁)` "tốt hơn" `(d₂, k₂)` nếu `d₁ > d₂`, hoặc `d₁ = d₂` và `k₁ > k₂` — chính là thứ tự tuple. Đáp án là phím của lần bấm **tốt nhất** theo thứ tự này.

**Invariant (cách một lượt):** sau khi xét các lần bấm `0..i`, `(best_dur, best_key)` là cặp tốt nhất trong số đó.
- `i = 0`: chỉ có một lần bấm ✓.
- Bước `i`: cặp mới tốt hơn kỷ lục thì thay, ngược lại giữ → kỷ lục vẫn là tốt nhất trong `0..i` ✓.

Hai lần bấm cùng phím và cùng thời lượng cho cùng tuple → chọn cái nào cũng ra cùng phím, không ảnh hưởng.

### 10.1. Độ phức tạp

Gọi `n = len(releaseTimes)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. List thời lượng + lọc | `O(n)` (3 lượt) | `O(n)` |
| 2. Một lượt | `O(n)` | **`O(1)`** |
| 3. `max` trên tuple | `O(n)` | `O(n)` (`prev`), có thể `O(1)` nếu dùng generator |
| 4. Dict theo phím | `O(n)` | `O(σ)` với `σ ≤ 26` |

## 11. Bộ test

1. Hai ví dụ đề.
2. Biên `n = 2`: phím đầu dài nhất; phím sau dài nhất; hai phím hòa.
3. Hòa nhiều lần, phím lớn nhất nằm ở đầu/giữa/cuối.
4. Phím đầu tiên dài nhất (dễ quên vì phải dùng `releaseTimes[0]`).
5. Giá trị lớn tới `10^9`.
6. Đối chiếu 9 hàm trên dữ liệu ngẫu nhiên (thời lượng nhỏ để hay hòa).

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual!r})")

solvers = [slowest_key_lists, slowest_key_one_pass, slowest_key_tuple, slowest_key_index, slowest_key_dict,
           Solution().slowestKey, SolutionTuple().slowestKey, SolutionLists().slowestKey, SolutionDict().slowestKey]
cases = [
    ("ví dụ 1", [9, 29, 49, 50], "cbcd", "c"),
    ("ví dụ 2", [12, 23, 36, 46, 62], "spuda", "a"),
    ("n=2, phím đầu dài nhất", [10, 11], "ab", "a"),
    ("n=2, phím sau dài nhất", [1, 11], "ba", "a"),
    ("n=2, hòa", [5, 10], "ab", "b"),
    ("hòa 3, lớn nhất ở đầu", [5, 10, 15], "zay", "z"),
    ("hòa 3, lớn nhất ở giữa", [5, 10, 15], "aza", "z"),
    ("cùng phím nhiều lần", [1, 3, 100], "aab", "b"),
    ("phím đầu dài nhất", [100, 101, 102], "xyz", "x"),
    ("giá trị lớn", [10**9 - 1, 10**9], "pq", "p"),
]
for name, times, keys, expected in cases:
    for f in solvers:
        assert f(times, keys) == expected, (name, f)
    check(name, slowest_key_one_pass(times, keys), expected)

ví dụ 1                          PASS ('c')
ví dụ 2                          PASS ('a')
n=2, phím đầu dài nhất           PASS ('a')
n=2, phím sau dài nhất           PASS ('a')
n=2, hòa                         PASS ('b')
hòa 3, lớn nhất ở đầu            PASS ('z')
hòa 3, lớn nhất ở giữa           PASS ('z')
cùng phím nhiều lần              PASS ('b')
phím đầu dài nhất                PASS ('x')
giá trị lớn                      PASS ('p')


In [18]:
import random
rng = random.Random(1629)

def brute_reference(times, keys):
    # Tham chiếu thô: duyệt mọi lần bấm, so sánh với mọi lần khác
    n = len(times)
    durs = [times[0]] + [times[i] - times[i - 1] for i in range(1, n)]
    for i in range(n):
        if all((durs[i], keys[i]) >= (durs[j], keys[j]) for j in range(n)):
            return keys[i]

for _ in range(800):
    n = rng.randint(2, 30)
    durs = [rng.randint(1, 4) for _ in range(n)]
    times, t = [], 0
    for d in durs:
        t += d
        times.append(t)
    keys = "".join(rng.choice("abcxyz") for _ in range(n))
    expected = brute_reference(times, keys)
    for f in solvers:
        assert f(times, keys) == expected
print("800 bộ dữ liệu ngẫu nhiên: 9 hàm khớp tham chiếu")

800 bộ dữ liệu ngẫu nhiên: 9 hàm khớp tham chiếu


## 12. Lỗi người mới hay gặp

### Lỗi 1: Coi `releaseTimes[i]` là thời lượng

Đó là **mốc thời gian**, luôn tăng → cách sai luôn chọn phím **cuối cùng**.

In [19]:
def wrong_use_release_time(times, keys):
    return max(zip(times, keys))[1]

print("sai:", wrong_use_release_time([9, 29, 49, 50], "cbcd"), "| đúng:", slowest_key_one_pass([9, 29, 49, 50], "cbcd"))

sai: d | đúng: c


### Lỗi 2: Bắt đầu vòng lặp từ `i = 1` mà quên phím đầu tiên

Khởi tạo `best_dur = 0, best_key = ""` rồi duyệt từ `i = 1` → không bao giờ xét lần bấm `0`.

In [20]:
def wrong_skip_first(times, keys):
    best_dur, best_key = 0, ""
    for i in range(1, len(times)):
        d = times[i] - times[i - 1]
        if d > best_dur or (d == best_dur and keys[i] > best_key):
            best_dur, best_key = d, keys[i]
    return best_key

print("sai:", wrong_skip_first([100, 101, 102], "xyz"), "| đúng:", slowest_key_one_pass([100, 101, 102], "xyz"))

sai: z | đúng: x


### Lỗi 3: Chỉ dùng `>` — quên luật phá hòa

Khi hòa sẽ giữ phím **xuất hiện trước**, không phải phím lớn nhất.

In [21]:
def wrong_no_tiebreak(times, keys):
    best_dur, best_key = times[0], keys[0]
    for i in range(1, len(times)):
        d = times[i] - times[i - 1]
        if d > best_dur:
            best_dur, best_key = d, keys[i]
    return best_key

print("sai:", wrong_no_tiebreak([9, 29, 49, 50], "cbcd"), "| đúng:", slowest_key_one_pass([9, 29, 49, 50], "cbcd"))

sai: b | đúng: c


### Lỗi 4: Cộng dồn thời lượng theo phím

Đề so **từng lần bấm**. Cộng dồn `c` (9 + 20 = 29) có thể cho kết quả đúng tình cờ nhưng sai ở ca khác.

In [22]:
def wrong_sum_per_key(times, keys):
    total, prev = {}, 0
    for t, k in zip(times, keys):
        total[k] = total.get(k, 0) + (t - prev)
        prev = t
    return max(total, key=lambda k: (total[k], k))

t, k = [1, 2, 3, 10], "aaab"   # a giữ 1+1+1 = 3 lần ngắn, b giữ 7
print("ca đúng tình cờ:", wrong_sum_per_key(t, k))
t, k = [3, 6, 9, 13], "aaab"   # mỗi lần a = 3, b = 4 -> đáp án b; cộng dồn a = 9
print("sai:", wrong_sum_per_key(t, k), "| đúng:", slowest_key_one_pass(t, k))

ca đúng tình cờ: b
sai: a | đúng: b


## 13. Mẹo nhận dạng pattern

- "Lớn nhất, nếu hòa thì chọn theo tiêu chí phụ" → gói thành **tuple** `(tiêu chí chính, tiêu chí phụ)` rồi `max`; hoặc một lượt với điều kiện `a > best or (a == best and b > best_b)`.
- Tiêu chí phụ ngược chiều (hòa thì chọn **nhỏ** hơn) → dùng `(d, -ord(k))` hoặc đảo dấu phần phụ.
- Cho **mốc thời gian** tăng dần mà hỏi **khoảng thời gian** → lấy **hiệu kề nhau** (đệm `0` ở đầu nếu bắt đầu từ 0).
- Bài liên quan: 495 Teemo Attacking (khoảng thời gian), 1854 Maximum Population Year, 2224 Minimum Number of Operations to Convert Time, 1491 Average Salary, 414 Third Maximum Number.

## 14. Biến thể

### 14.1. Phím **nhanh nhất**, hòa thì chọn phím **nhỏ** nhất

Lấy `min` trên tuple `(d, k)` — vừa ngắn nhất vừa nhỏ nhất khi hòa.

In [23]:
def fastest_key(times, keys):
    prev = [0] + times[:-1]
    return min(zip((c - p for c, p in zip(times, prev)), keys))[1]

print(fastest_key([9, 29, 49, 50], "cbcd"), fastest_key([5, 10, 15], "zay"))   # d a

d a


### 14.2. Dài nhất, nhưng hòa thì chọn phím **nhỏ** nhất

Đảo dấu tiêu chí phụ: so theo `(d, -ord(k))`.

In [24]:
def slowest_key_small_tie(times, keys):
    prev = [0] + times[:-1]
    best = max(zip((c - p for c, p in zip(times, prev)), keys), key=lambda dk: (dk[0], -ord(dk[1])))
    return best[1]

print(slowest_key_small_tie([9, 29, 49, 50], "cbcd"))   # b

b


### 14.3. Tổng thời gian giữ của từng phím (thống kê)

In [25]:
def total_per_key(times, keys):
    total, prev = {}, 0
    for t, k in zip(times, keys):
        total[k] = total.get(k, 0) + (t - prev)
        prev = t
    return dict(sorted(total.items()))

print(total_per_key([9, 29, 49, 50], "cbcd"))

{'b': 20, 'c': 29, 'd': 1}


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** `releaseTimes = [4, 9, 14, 15]`, `keysPressed = "abxa"`. Viết thời lượng từng lần và đáp án.

**Bài 2 — đoán output.** `max([(3, 'b'), (3, 'a'), (2, 'z')])` và `(5, 'a') < (5, 'b')` cho gì?

### Bài 3 — Điền TODO: lời giải một lượt

In [26]:
def exercise_slowest_key(releaseTimes: list[int], keysPressed: str) -> str:
    # TODO 1: khởi tạo kỷ lục bằng lần bấm 0
    # TODO 2: duyệt i từ 1, d = releaseTimes[i] - releaseTimes[i-1]
    # TODO 3: thay kỷ lục nếu d lớn hơn, hoặc bằng mà phím lớn hơn
    pass

# assert exercise_slowest_key([9, 29, 49, 50], "cbcd") == "c"

### Bài 4 — Tự viết hàm trả list thời lượng bằng `zip` (không dùng chỉ số)

In [27]:
def exercise_durations(releaseTimes: list[int]) -> list[int]:
    # TODO: zip([0] + releaseTimes, releaseTimes)
    pass

# assert exercise_durations([9, 29, 49, 50]) == [9, 20, 20, 1]

### Bài 5 — Biến thể: trả về **chỉ số** lần bấm chậm nhất (hòa thì lấy chỉ số **nhỏ** nhất)

In [28]:
def exercise_slowest_index(releaseTimes: list[int]) -> int:
    # TODO: chỉ thay khi d > best (dấu > nghiêm ngặt giữ chỉ số đầu tiên)
    pass

# assert exercise_slowest_index([9, 29, 49, 50]) == 1

### Bài 6 — Mini challenge: top-3 phím theo thời lượng dài nhất của mỗi phím

Với mỗi phím lấy thời lượng **dài nhất** của nó; trả về 3 phím đứng đầu theo luật (thời lượng giảm dần, hòa thì phím lớn trước). Ít hơn 3 phím thì trả hết.

In [29]:
def exercise_top3_keys(releaseTimes: list[int], keysPressed: str) -> list[str]:
    # TODO: dict max theo phím rồi sorted với key=(dur, k), reverse=True
    pass

# assert exercise_top3_keys([9, 29, 49, 50], "cbcd") == ["c", "b", "d"]

## 16. Gợi ý 3 mức

**Bài 1:** M1: lần đầu = mốc đầu. M2: `4, 5, 5, 1`. M3: hòa 5 giữa `b` và `x`.

**Bài 2:** M1: so phần tử đầu trước. M2: hòa 3 → so `'b'` với `'a'`. M3: `'a' < 'b'`.

**Bài 3:** M1: xem mục 5. M2: `best_dur, best_key = releaseTimes[0], keysPressed[0]`. M3: `if d > best_dur or (d == best_dur and k > best_key)`.

**Bài 4:** M1: mốc trước của lần đầu là 0. M2: `zip([0] + t, t)`. M3: `[c - p for p, c in zip(...)]`.

**Bài 5:** M1: kỷ lục là chỉ số. M2: bắt đầu `best_i = 0`. M3: so `d > durs[best_i]` (nghiêm ngặt).

**Bài 6:** M1: giống `slowest_key_dict`. M2: `sorted(longest, key=lambda k: (longest[k], k), reverse=True)`. M3: lấy `[:3]`.

## 17. Đáp án tham khảo

**Bài 1:** thời lượng `[4, 5, 5, 1]`; dài nhất 5 của `b` và `x`; `'x' > 'b'` → `"x"`.

**Bài 2:** `(3, 'b')` và `True`.

In [30]:
def answer_slowest_key(releaseTimes, keysPressed):
    best_dur, best_key = releaseTimes[0], keysPressed[0]
    for i in range(1, len(releaseTimes)):
        d, k = releaseTimes[i] - releaseTimes[i - 1], keysPressed[i]
        if d > best_dur or (d == best_dur and k > best_key):
            best_dur, best_key = d, k
    return best_key

def answer_durations(releaseTimes):
    return [c - p for p, c in zip([0] + releaseTimes, releaseTimes)]

def answer_slowest_index(releaseTimes):
    durs = answer_durations(releaseTimes)
    best_i = 0
    for i in range(1, len(durs)):
        if durs[i] > durs[best_i]:
            best_i = i
    return best_i

def answer_top3_keys(releaseTimes, keysPressed):
    longest = {}
    for d, k in zip(answer_durations(releaseTimes), keysPressed):
        longest[k] = max(longest.get(k, 0), d)
    return sorted(longest, key=lambda k: (longest[k], k), reverse=True)[:3]

assert answer_durations([4, 9, 14, 15]) == [4, 5, 5, 1] and answer_slowest_key([4, 9, 14, 15], "abxa") == "x"
assert max([(3, 'b'), (3, 'a'), (2, 'z')]) == (3, 'b') and ((5, 'a') < (5, 'b')) is True
assert answer_slowest_key([9, 29, 49, 50], "cbcd") == "c"
assert answer_durations([9, 29, 49, 50]) == [9, 20, 20, 1]
assert answer_slowest_index([9, 29, 49, 50]) == 1
assert answer_top3_keys([9, 29, 49, 50], "cbcd") == ["c", "b", "d"]
for _ in range(300):
    n = rng.randint(2, 20)
    times, t = [], 0
    for _ in range(n):
        t += rng.randint(1, 3)
        times.append(t)
    keys = "".join(rng.choice("abcd") for _ in range(n))
    assert answer_slowest_key(times, keys) == brute_reference(times, keys)
    durs = answer_durations(times)
    assert durs[answer_slowest_index(times)] == max(durs) and answer_slowest_index(times) == durs.index(max(durs))
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def slowestKey(self, releaseTimes: list[int], keysPressed: str) -> str:
        best_dur, best_key = releaseTimes[0], keysPressed[0]   # lần 0: mốc trước = 0
        for i in range(1, len(releaseTimes)):
            d = releaseTimes[i] - releaseTimes[i - 1]          # thời lượng = hiệu kề nhau
            if d > best_dur or (d == best_dur and keysPressed[i] > best_key):
                best_dur, best_key = d, keysPressed[i]
        return best_key
```

**Câu thần chú:** *"Mốc → hiệu; so (thời lượng, phím) như tuple; nhớ lần bấm đầu."*

**Checklist trước khi submit:**
- [ ] Thời lượng = `releaseTimes[i] - releaseTimes[i-1]`, lần đầu = `releaseTimes[0]`?
- [ ] Đã xét **lần bấm đầu tiên**?
- [ ] Có luật phá hòa: bằng thời lượng thì lấy phím **lớn** hơn?
- [ ] So từng lần bấm, không cộng dồn theo phím?
- [ ] Trả về **phím** (chuỗi 1 ký tự), tên method `slowestKey`?